# Consolidated sweep - ingest many tables in ONE job task

One `lakefed_ingest_mt_sweep` run processes a whole **task_collection** of (tenant db, table)
tasks in this single notebook, using a thread pool + Spark **FAIR scheduler pools** so each
table's work overlaps on the cluster. This decouples throughput from the workspace job limits
(2,000 concurrent task runs; 10,000 run-submits/hour) that a task-per-table design hits - the
throttle becomes compute (cluster cores), which is the intended cost model. Shard by
`task_collection` for horizontal scale: run one sweep per collection.

Notes:
- Cluster must run the **FAIR** scheduler (`spark.scheduler.mode=FAIR`) for the pools to apply.
- All Change-Tracking / source-touching work is in **ct_engine.py** (Alex, workstreams A/B/F) -
  this notebook is pure orchestration + Lakebase checkpointing + telemetry.
- Because the per-table work runs in-process (no Jobs-UI task per table), progress/telemetry is
  written to Lakebase: one `sweep_run` row per run and one `ingest_event` row per (sweep, table),
  queryable live and read directly from the Lakebase UC catalog (see docs/monitoring.md).
- Per-table failures are isolated (one bad table does not fail the collection) and recorded to
  both `ingest_event` (with the stage it failed at) and the `ct_checkpoint` status for triage.

In [ ]:
# Cluster task. Bootstrap: put this notebook's dir on sys.path and import the shared modules.
import sys, os, json

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)

import state_store as ss
from state_store import StateStore
import ct_engine
import parallel

In [ ]:
# Params + connect to Lakebase (SDK-minted OAuth token; no stored password), then open a sweep.
def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


task_collection = widget("task_collection", "example_cohort")
db_instance = widget("db_instance", "lakefed-ingest-mt")
db_name = widget("db_name", "lakefed_ingest_mt")
parallelism = int(widget("parallelism", "16"))

store = StateStore.connect_lakebase(db_instance, db_name)
configs = store.list_task_configs(task_collection)


def _safe(fn, default=""):
    try:
        return fn() or default
    except Exception:
        return default


job_run_id = _safe(lambda: str(ctx.jobRunId().get()))
cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId", "")
sweep_id = store.start_sweep(task_collection, job_run_id, cluster_id, parallelism)
print(f"sweep {sweep_id}: collection {task_collection} -> {len(configs)} tables (parallelism={parallelism})")

In [ ]:
# Per-(tenant db, table) state machine with Lakebase telemetry. CT/source ops are ct_engine
# placeholders (Alex, A/B/F); everything here is orchestration + checkpointing + telemetry.
# The `stage` variable records how far a table got, so a failure pinpoints read vs merge vs
# checkpoint. `action` is provisional (seed vs increment) from phase; a cdc->reseed is still
# counted in the sweep roll-up (reseed_enqueued) and the reseed_queue table.
from functools import partial


def ingest_one(cfg, sweep_id):
    db, schema, table = cfg["src_database"], cfg["src_schema"], cfg["src_table"]
    sink = f"{cfg['sink_catalog']}.{cfg['sink_schema']}.{cfg['sink_table']}"
    pool = spark.sparkContext.getLocalProperty("spark.scheduler.pool") or ""

    cp = store.get_checkpoint(db, schema, table)
    phase = cp["phase"]
    v_from = cp["ct_version"]
    action = "seed" if ss.decide_action(phase, v_from, None) == "seed" else "increment"
    event_id = store.record_event_start(sweep_id, cfg, action, phase, pool)

    stage = "ensure_sink"
    rows_read = rows_merged = v_to = None
    try:
        ct_engine.ensure_sink_table(spark, cfg, sink)

        if action == "seed":
            stage = "read"
            v0 = ct_engine.current_version(cfg)              # capture BEFORE the seed
            stage = "seed"
            seeded = ct_engine.seed(spark, cfg, sink)
            rows_merged = seeded if isinstance(seeded, int) else None
            stage = "checkpoint"
            store.advance_version(db, schema, table, ct_version=v0, phase=ss.CDC, detail="seed complete")
            v_to = v0
            outcome = "seeded"
        else:
            stage = "read"
            mv = ct_engine.min_valid_version(cfg)
            if ss.decide_action(phase, v_from, mv) == "reseed":
                stage = "checkpoint"
                store.enqueue_reseed(db, schema, table, reason="retention_miss")  # flips phase -> seeding
                outcome = "reseed_enqueued"
            else:
                new_version = ct_engine.current_version(cfg)  # upper bound BEFORE the read
                stage = "merge"
                merged = ct_engine.read_and_merge_ct(spark, cfg, sink, v_from)
                rows_merged = merged if isinstance(merged, int) else None
                stage = "checkpoint"
                store.advance_version(db, schema, table, ct_version=new_version)  # only AFTER commit
                v_to = new_version
                outcome = "incremented"

        store.record_event_finish(event_id, "ok", stage=stage, rows_read=rows_read,
                                  rows_merged=rows_merged, ct_version_from=v_from, ct_version_to=v_to)
        print(f"[ok:{outcome}] {db}.{schema}.{table} pool={pool}")
        return outcome
    except Exception as e:
        err = str(e)
        try:
            store.record_event_finish(event_id, "failed", stage=stage, rows_read=rows_read,
                                      rows_merged=rows_merged, ct_version_from=v_from,
                                      ct_version_to=v_to, error=err)
        except Exception:
            pass
        try:
            store.set_status(db, schema, table, "failed", err[:500])
        except Exception:
            pass
        print(f"[failed@{stage}] {db}.{schema}.{table}: {err[:200]}")
        raise


worker = partial(ingest_one, sweep_id=sweep_id)


def set_pool(pool):
    # Each table's Spark jobs (e.g. its MERGE) run in their own FAIR pool so they overlap fairly.
    spark.sparkContext.setLocalProperty("spark.scheduler.pool", pool)

In [ ]:
# Sweep the collection: one thread per concurrency slot, round-robin FAIR pools, failures isolated.
results = parallel.run_parallel(
    configs, worker, parallelism, set_pool=set_pool, item_id=lambda c: c["id"]
)
summary = parallel.summarize(results)
by = summary.get("by_outcome", {})
reseeded = by.get("reseed_enqueued", 0)
skipped = by.get("skipped", 0)  # reserved for a future skip-no-change outcome
store.finish_sweep(
    sweep_id,
    total=summary["total"], ok=summary["ok"], failed=summary["failed"],
    skipped=skipped, reseeded=reseeded,
    status="completed" if summary["failed"] == 0 else "failed",
)

summary["sweep_id"] = sweep_id
summary["task_collection"] = task_collection
print(json.dumps(summary, indent=2))

failures = [r for r in results if r["status"] == "failed"]
if failures:
    print("failures:", json.dumps(failures, default=str)[:2000])

dbutils.notebook.exit(json.dumps(summary, default=str))